# databricks_launchpad_widgets\n
Generated from 00_common/databricks_launchpad_widgets.py for Databricks notebook import.\n

In [ ]:
"""Databricks UI widget helper driven by config/widget_options.yaml."""

from pathlib import Path
from typing import Any
import yaml

# dbutils compatibility shim
try:
    dbutils  # type: ignore[name-defined]  # noqa: F821
except NameError:
    class _NoOpWidgets:
        _values: dict[str, str] = {}
        def dropdown(self, name: str, default: str, choices: list[str], label: str = "") -> None:
            self._values[name] = default
        def get(self, name: str) -> str:
            return self._values.get(name, "")
        def removeAll(self) -> None:
            self._values.clear()
    class _NoOpDbutils:
        widgets = _NoOpWidgets()
    dbutils = _NoOpDbutils()  # type: ignore[assignment]

def load_widget_options(widget_file: str) -> dict:
    path = Path(widget_file)
    if not path.exists():
        raise FileNotFoundError(f"Widget options file not found: {path}")
    with open(path, "r", encoding="utf-8") as f:
        payload = yaml.safe_load(f) or {}
    return payload.get("widget_options", {})

def _choices(options: dict, key: str) -> list[str]:
    values = options.get(key, [])
    return [str(v) for v in values] if isinstance(values, list) else []

def _first(choices: list[str], fallback: str = "") -> str:
    return choices[0] if choices else fallback

def create_launchpad_widgets(widget_file: str | None = None) -> dict[str, Any]:
    if widget_file is None:
        widget_file = str(Path(__file__).resolve().parents[2] / "config" / "widget_options.yaml")
    options = load_widget_options(widget_file)
    try:
        dbutils.widgets.removeAll()
    except Exception:
        pass
    env_choices = _choices(options, "environments")
    bool_choices = _choices(options, "boolean_values")
    folder_choices = _choices(options, "metadata_folder_name")
    mount_choices = _choices(options, "metadata_base_mount_point")
    paths_choices = _choices(options, "paths_config_path")
    source_choices = _choices(options, "ikea_dropzone_mounts")
    dbutils.widgets.dropdown("environment", _first(env_choices, "dev"), env_choices, "Environment")
    dbutils.widgets.dropdown("full_load", _first(bool_choices, "False"), bool_choices, "Full Load")
    dbutils.widgets.dropdown("metadata_folder", _first(folder_choices), folder_choices, "Metadata Folder")
    dbutils.widgets.dropdown("metadata_mount", _first(mount_choices), mount_choices, "Metadata Base Mount")
    dbutils.widgets.dropdown("paths_config_path", _first(paths_choices), paths_choices, "Paths Config")
    dbutils.widgets.dropdown("source_mount", _first(source_choices), source_choices, "Source Mount")
    return options

_WIDGET_NAMES = [
    "environment",
    "full_load",
    "metadata_folder",
    "metadata_mount",
    "paths_config_path",
    "source_mount",
]

def get_widget_values() -> dict[str, str]:
    return {name: dbutils.widgets.get(name) for name in _WIDGET_NAMES}